# 2주차 — 코드 없이 AI 만들기 (Teachable Machine)
### 인공지능 플랫폼실습 · 인공지능소프트웨어과

---

## 이번 주 학습 목표
1. 코드 없이 나만의 이미지 분류 AI를 만들 수 있다.
2. '학습(training)'과 '추론(inference)'의 차이를 설명할 수 있다.
3. Teachable Machine에서 내보낸 모델을 Colab에서 불러와 사용할 수 있다.

## 오늘의 흐름
```
[1] 브라우저(Teachable Machine)         [2] Colab (이 노트북)
    사진 모으기 → 학습(Train)      →       내보낸 모델 불러오기 → 추론(Predict)
    ── 코드 0줄 ──                        ── 코드로 사용하기 ──
```

## 이 노트북을 열기 전에 (브라우저에서 먼저)
1. <https://teachablemachine.withgoogle.com/> 접속 → **이미지 프로젝트** 시작
2. 물건 2~3가지를 클래스로 만들고 사진을 각각 50장 이상 찍기
3. **모델 학습시키기(Train Model)** 클릭
4. **모델 내보내기(Export Model)** → **Tensorflow Lite** 탭 → **부동 소수점(Floating point)** → **모델 다운로드**
5. 받은 `converted_tflite.zip` 을 준비 (안에 `model_unquant.tflite` 와 `labels.txt` 가 들어 있습니다)

> ⏱ 예상 소요 시간: 브라우저 작업 60분 + 이 노트북 40분
> ※ Teachable Machine은 **데스크톱 크롬(Chrome) 또는 사파리**에서만 동작합니다. (휴대폰 브라우저 ✕)


---
# STEP 1. 내보낸 모델 파일 올리기  (약 10분)

Teachable Machine에서 받은 zip 파일을 Colab으로 올립니다.


In [ ]:
# ============================================================
# [실습 1-1] zip 파일 업로드하기   ※ Colab 전용
# ------------------------------------------------------------
# 실행하면 [파일 선택] 버튼이 나타납니다.
# 아까 다운로드한 converted_tflite.zip 을 고르세요.
# 업로드가 끝날 때까지 다른 셀을 실행하지 마세요. (진행 표시가 100%가 될 때까지)
# ============================================================

from google.colab import files

uploaded = files.upload()          # 파일 선택 창이 뜹니다
print("\n업로드된 파일 :", list(uploaded.keys()))


In [ ]:
# ============================================================
# [실습 1-2] zip 압축 풀기
# ------------------------------------------------------------
# zipfile 은 압축 파일을 다루는 파이썬 기본 도구입니다.
# glob 은 "이런 이름의 파일 찾아줘"라고 할 때 씁니다.
# ============================================================

import zipfile, glob, os

# 현재 폴더에 있는 zip 파일을 모두 찾습니다
zips = glob.glob("*.zip")
print("찾은 zip 파일 :", zips)

for z in zips:
    with zipfile.ZipFile(z) as f:
        f.extractall(".")          # 현재 폴더에 압축을 풉니다
        print(f"{z} -> 압축 해제 완료:", f.namelist())

# 압축을 푼 뒤 폴더에 어떤 파일이 있는지 확인합니다
print("\n현재 폴더 파일 목록 :")
for name in sorted(os.listdir(".")):
    if not name.startswith("."):
        print("  -", name)


### ✅ 확인 체크포인트
목록에 **`.tflite` 로 끝나는 파일**과 **`labels.txt`** 가 보이면 성공입니다.
(파일 이름은 `model_unquant.tflite` 또는 `model.tflite` 일 수 있습니다)


---
# STEP 2. 클래스 이름(정답표) 읽기  (약 5분)

`labels.txt` 에는 내가 만든 클래스 이름이 순서대로 적혀 있습니다.
AI는 이름을 모르고 **번호(0, 1, 2 …)로만** 답합니다. 그 번호를 이름으로 바꿔주는 표가 이 파일입니다.


In [ ]:
# ============================================================
# [실습 2-1] labels.txt 읽어서 클래스 이름 목록 만들기
# ------------------------------------------------------------
# labels.txt 의 한 줄은 "0 볼펜" 처럼 [번호][공백][이름] 형태입니다.
# split(" ", 1) 은 첫 번째 공백에서 한 번만 잘라 [번호, 이름] 으로 나눕니다.
# ============================================================

with open("labels.txt", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

labels = []
for line in lines:
    parts = line.split(" ", 1)          # 최대 1번만 자르기
    name = parts[1] if len(parts) > 1 else parts[0]
    labels.append(name)

print("클래스 개수 :", len(labels))
for i, name in enumerate(labels):
    print(f"  {i}번 -> {name}")


---
# STEP 3. 모델 불러오기  (약 10분)

내보낸 모델은 **TensorFlow Lite** 형식입니다.
휴대폰이나 작은 기기에서도 돌아가도록 가볍게 만든 형태라고 생각하면 됩니다.


In [ ]:
# ============================================================
# [실습 3-1] 모델을 실행할 도구 설치
# ------------------------------------------------------------
# 예전에는 tf.lite.Interpreter 를 썼지만 지원이 끝나가고 있어서,
# 구글이 새로 권장하는 ai-edge-litert 를 씁니다.
#   -q : 설치 메시지를 조용히
# ============================================================

!pip install -q ai-edge-litert

print("설치 완료")


In [ ]:
# ============================================================
# [실습 3-2] 모델 파일을 찾아서 불러오기
# ------------------------------------------------------------
# ★ 파일 이름을 직접 적지 않고 '찾아서' 쓰는 이유:
#   Teachable Machine 버전에 따라 model.tflite / model_unquant.tflite 로
#   이름이 다르게 나오기 때문입니다.
# ============================================================

import glob
import numpy as np

# ai_edge_litert 가 없으면 기존 tensorflow 방식으로 자동 대체합니다
try:
    from ai_edge_litert.interpreter import Interpreter
    print("LiteRT 사용")
except ImportError:
    from tensorflow.lite import Interpreter
    print("tensorflow.lite 사용 (대체 경로)")

# .tflite 파일 찾기
tflite_files = glob.glob("**/*.tflite", recursive=True)
if len(tflite_files) == 0:
    raise FileNotFoundError(
        "tflite 파일이 없습니다. STEP 1의 업로드/압축 해제를 다시 하세요.")

MODEL_PATH = tflite_files[0]
print("사용할 모델 :", MODEL_PATH)

# 모델 준비
interpreter = Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()          # 메모리 자리 잡기 (필수)

inp = interpreter.get_input_details()[0]    # 입력 정보
out = interpreter.get_output_details()[0]   # 출력 정보

IMG_H, IMG_W = int(inp["shape"][1]), int(inp["shape"][2])

print("\n[모델이 요구하는 입력]")
print("  크기   :", IMG_W, "x", IMG_H, "픽셀")
print("  자료형 :", inp["dtype"].__name__)
print("[모델이 내놓는 출력]")
print("  모양   :", out["shape"], "-> 클래스", int(out["shape"][1]), "개의 확률")


> 💡 **왜 224 x 224 인가요?**
> Teachable Machine이 쓰는 모델이 그 크기로 학습됐기 때문입니다.
> 그래서 우리가 넣을 사진도 **같은 크기로 줄여서** 넣어야 합니다. 다음 단계에서 합니다.


---
# STEP 4. 사진 한 장 예측하기 (추론)  (약 15분)

드디어 **추론(inference)** 입니다.
학습은 Teachable Machine이 이미 끝냈고, 우리는 완성된 AI에게 질문만 던지는 것입니다.


In [ ]:
# ============================================================
# [실습 4-1] 테스트할 사진 올리기   ※ Colab 전용
# ------------------------------------------------------------
# 학습에 쓰지 않은 '새로운 사진'을 올려야 제대로 된 시험이 됩니다.
# ============================================================

from google.colab import files

uploaded = files.upload()
TEST_IMAGE = list(uploaded.keys())[0]
print("테스트할 사진 :", TEST_IMAGE)


In [ ]:
# ============================================================
# [실습 4-2] 사진을 모델이 먹을 수 있는 형태로 바꾸기 (전처리)
# ------------------------------------------------------------
# 사람은 사진을 그냥 보지만, AI는 '숫자 덩어리'만 받을 수 있습니다.
# 그래서 아래 3단계를 거칩니다.
#   1) 크기 맞추기   : 224 x 224 로 줄이기
#   2) 숫자로 바꾸기 : 각 픽셀을 0~255 숫자로
#   3) 범위 맞추기   : -1 ~ 1 사이로 (Teachable Machine 모델의 규칙)
# ============================================================

from PIL import Image          # 사진을 다루는 도구
import numpy as np             # 숫자 덩어리를 다루는 도구

def 사진_준비하기(파일이름):
    """사진 파일을 모델에 넣을 수 있는 숫자 덩어리로 바꿔서 돌려준다"""
    img = Image.open(파일이름).convert("RGB")     # 흑백/투명 사진도 컬러로 통일
    img = img.resize((IMG_W, IMG_H))              # 1) 크기 맞추기

    arr = np.asarray(img, dtype=np.float32)       # 2) 숫자로 바꾸기 (0~255)
    arr = (arr / 127.5) - 1.0                     # 3) 범위 맞추기 (-1~1)

    arr = np.expand_dims(arr, axis=0)             # 사진 1장짜리 '묶음'으로 만들기
    return arr

X = 사진_준비하기(TEST_IMAGE)
print("모델에 넣을 데이터 모양 :", X.shape)
print("값의 범위 : 최소", round(float(X.min()), 2), "~ 최대", round(float(X.max()), 2))


In [ ]:
# ============================================================
# [실습 4-3] 예측하기 — 오늘의 핵심 3줄
# ------------------------------------------------------------
#   set_tensor : 준비한 사진을 모델에 넣고
#   invoke     : 계산을 시키고
#   get_tensor : 결과를 꺼낸다
# ============================================================

interpreter.set_tensor(inp["index"], X)     # 넣고
interpreter.invoke()                        # 돌리고
probs = interpreter.get_tensor(out["index"])[0]   # 꺼내고

best = int(np.argmax(probs))                # 확률이 가장 높은 번호
print("=" * 34)
print(f" 예측 : {labels[best]}  ({probs[best]*100:.1f}%)")
print("=" * 34)

print("\n[전체 확률]")
for name, p in sorted(zip(labels, probs), key=lambda t: -t[1]):
    bar = "■" * int(p * 20)                 # 확률을 막대로 그려보기
    print(f"  {name:10s} {p*100:5.1f}%  {bar}")


> 💡 **확률을 모두 더하면 100%가 됩니다.**
> AI는 "이거다!"라고 딱 잘라 말하지 않고 **"이럴 가능성이 몇 %"** 라고 답합니다.
> 그래서 학습에 없던 물건을 보여줘도, AI는 배운 것 중에서 가장 비슷한 것을 억지로 고릅니다.
> → 이 성질이 오늘의 과제 2번과 이어집니다.


---
# STEP 5. 결과를 그래프로 보기  (약 10분)

1주차에 배운 한글 폰트 설정을 그대로 씁니다.


In [ ]:
# ============================================================
# [실습 5-1] 한글 폰트 설치 + 적용  ※ 1주차와 같은 코드 (1~2분)
# ------------------------------------------------------------
# 설치와 등록을 한 셀에 함께 넣어야 FileNotFoundError 가 나지 않습니다.
# ============================================================

!apt-get install -y -qq fonts-nanum

import glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_files = glob.glob("/usr/share/fonts/**/Nanum*.ttf", recursive=True)
if len(font_files) == 0:
    print("❌ 폰트를 찾지 못했습니다. 이 셀을 한 번 더 실행하세요.")
else:
    FONT = ([p for p in font_files if p.endswith("NanumGothic.ttf")]
            or sorted(font_files))[0]
    fm.fontManager.addfont(FONT)
    plt.rcParams["font.family"] = fm.FontProperties(fname=FONT).get_name()
    plt.rcParams["axes.unicode_minus"] = False
    print("✅ 한글 폰트 준비 완료 :", plt.rcParams["font.family"])


In [ ]:
# ============================================================
# [실습 5-2] 예측 확률을 막대그래프로 그리기
# ------------------------------------------------------------
# barh( ) 는 가로 막대그래프입니다. 이름이 길어도 안 겹쳐서 보기 좋습니다.
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.barh(labels, probs * 100)            # 확률을 % 로 바꿔서 그리기
plt.xlim(0, 100)
plt.xlabel("확률 (%)")
plt.title("AI의 예측 결과")

# 막대 끝에 숫자 붙이기
for i, p in enumerate(probs * 100):
    plt.text(p + 1, i, f"{p:.1f}%", va="center", fontsize=10)

plt.tight_layout()
plt.savefig("예측결과.png", dpi=100, bbox_inches="tight")
plt.show()

print("그래프를 이미지 파일(예측결과.png)로도 저장했습니다.")


---
# STEP 6. 여러 장 한꺼번에 예측하기  (약 10분)

한 장씩 하면 느립니다. 폴더에 있는 사진을 **반복문으로 한 번에** 처리해 봅시다.


In [ ]:
# ============================================================
# [실습 6-1] 폴더 안의 모든 사진을 예측하기
# ------------------------------------------------------------
# 함수로 만들어 두면 사진이 몇 장이든 같은 코드로 처리할 수 있습니다.
# ============================================================

import glob

def 예측하기(파일이름):
    """사진 파일 하나를 받아 (예측이름, 확률) 을 돌려준다"""
    X = 사진_준비하기(파일이름)
    interpreter.set_tensor(inp["index"], X)
    interpreter.invoke()
    p = interpreter.get_tensor(out["index"])[0]
    i = int(np.argmax(p))
    return labels[i], float(p[i])

# 현재 폴더의 사진 파일을 모두 찾습니다
사진목록 = sorted(glob.glob("*.jpg") + glob.glob("*.png") + glob.glob("*.jpeg"))

# 우리가 STEP 5에서 저장한 그래프 이미지는 사진이 아니므로 제외합니다
사진목록 = [f for f in 사진목록 if f != "예측결과.png"]

print(f"찾은 사진 {len(사진목록)}장\n")

for 파일 in 사진목록:
    이름, 확률 = 예측하기(파일)
    표시 = "✅" if 확률 >= 0.8 else "❓"     # 확률이 낮으면 물음표
    print(f"{표시} {파일:22s} -> {이름:10s} ({확률*100:.1f}%)")

print("\n※ ❓ 표시는 AI도 확신이 없다는 뜻입니다. 왜 그런지 생각해 보세요.")


---
# (참고) Keras `.h5` 로 내보냈다면

Teachable Machine에서 **Tensorflow → Keras** 로 내보내면 `keras_model.h5` 가 받아집니다.
이 파일은 오래된 형식이라 요즘 Colab에서 그냥 열면 **오류가 납니다.**

```
TypeError: Error when deserializing class 'DepthwiseConv2D' using config={... 'groups': 1 ...}
ValueError: Unrecognized keyword arguments passed to DepthwiseConv2D: {'groups': 1}
```

아래처럼 `groups` 를 무시하는 층으로 바꿔치기하면 열립니다.
**수업에서는 TFLite 방식(STEP 3)을 쓰므로 이 셀은 실행하지 않아도 됩니다.**


In [ ]:
# ============================================================
# [참고] Keras .h5 모델을 여는 방법  ※ 파일이 있을 때만 실행
# ------------------------------------------------------------
# DepthwiseConv2D 층이 예전에는 groups 라는 설정을 갖고 있었는데
# 지금 버전에서는 없어졌습니다. 그래서 "모르는 설정"이라며 오류를 냅니다.
# -> groups 를 받아도 그냥 무시하는 새 층을 만들어 대신 쓰게 합니다.
# ============================================================

import os

if os.path.exists("keras_model.h5"):
    from keras.models import load_model
    from keras.layers import DepthwiseConv2D

    class FixedDepthwiseConv2D(DepthwiseConv2D):
        def __init__(self, *args, groups=1, **kwargs):
            # groups 를 받아서 그냥 버립니다
            super().__init__(*args, **kwargs)

    model = load_model("keras_model.h5", compile=False,
                       custom_objects={"DepthwiseConv2D": FixedDepthwiseConv2D})
    print("Keras 모델 불러오기 성공 :", model.output_shape)
else:
    print("keras_model.h5 파일이 없어 건너뜁니다. (TFLite 방식만 써도 충분합니다)")


---
# 오늘의 과제  (약 20분)


In [ ]:
# ============================================================
# [과제 1] 내 모델의 성적표 만들기
# ------------------------------------------------------------
# TODO 1) 각 클래스마다 '학습에 쓰지 않은' 새 사진을 2장씩 올리세요.
# TODO 2) 아래 정답 목록을 본인 사진에 맞게 채우세요.
# TODO 3) 실행해서 몇 개를 맞혔는지 확인하세요.
# ============================================================

# {"파일이름": "정답 클래스 이름"} 형태로 적습니다
정답표 = {
    "TODO_사진1.jpg": "TODO_정답이름",
    "TODO_사진2.jpg": "TODO_정답이름",
    # 필요한 만큼 줄을 추가하세요
}

맞은개수 = 0
for 파일, 정답 in 정답표.items():
    try:
        예측이름, 확률 = 예측하기(파일)
    except FileNotFoundError:
        print(f"⚠ {파일} 파일이 없습니다. 파일 이름을 확인하세요.")
        continue
    맞음 = (예측이름 == 정답)
    맞은개수 += int(맞음)
    print(f"{'⭕' if 맞음 else '❌'} {파일} | 정답 {정답} / 예측 {예측이름} ({확률*100:.1f}%)")

if len(정답표) > 0:
    print(f"\n정확도 : {맞은개수}/{len(정답표)} = {맞은개수/len(정답표)*100:.0f}%")


### [과제 2] AI를 속여 보기 (생각해 보기)

학습에 **전혀 없던 물건**(예: 손, 책, 얼굴)을 찍어서 예측해 보세요.

1. AI는 뭐라고 답했나요? 확률은 몇 %였나요?
2. AI가 "모르겠다"고 답하지 못하는 이유는 무엇일까요?
3. 이런 성질 때문에 생길 수 있는 문제를 한 가지만 적어보세요.

> ✍️ 아래 칸을 두 번 클릭해서 답을 적으세요.

**1번 답 :**

**2번 답 :**

**3번 답 :**


---
# 자주 나는 오류와 해결법

| 증상(오류 메시지) | 원인 | 해결 방법 |
|---|---|---|
| `FileNotFoundError: labels.txt` | 압축을 풀지 않음 | [실습 1-2] 를 실행 |
| `tflite 파일이 없습니다` | 업로드 실패 / 다른 형식으로 내보냄 | TM에서 **Tensorflow Lite → 부동 소수점** 으로 다시 내보내기 |
| `ValueError: Cannot set tensor: Dimension mismatch` | 사진 크기를 안 맞춤 | `사진_준비하기( )` 를 거쳐서 넣기 |
| 예측이 전부 같은 클래스 | 클래스별 사진 수가 너무 적거나 배경이 똑같음 | 각 클래스 50장 이상, 배경·각도 바꿔서 다시 촬영 |
| 확률이 40% 근처에서 헤맴 | 물건들이 서로 너무 비슷함 | 생김새가 뚜렷이 다른 물건으로 교체 |
| `TypeError: ... DepthwiseConv2D ... 'groups': 1` | 오래된 `.h5` 를 그냥 열었음 | 위 (참고) 셀의 방법 사용 |
| 세션이 끊겨 변수가 사라짐 | 장시간 방치 | [런타임] → [모두 실행] |

> 🙋 **막혔을 때** ① 오류 메시지 맨 아랫줄 읽기 → ② 위 표에서 찾기
> → ③ 옆 사람 화면과 비교 → ④ 손 들어 교수 부르기

---

# 더 알아보기

| 자료 | 주소 |
|---|---|
| Teachable Machine 공식 | https://teachablemachine.withgoogle.com/ |
| 생활코딩 — Teachable Machine 강의 | https://opentutorials.org/course/4548/28897 |
| 모두의연구소 — 노코드 AI 활용하기 | https://modulabs.co.kr/blog/teachable-machine |
| LiteRT(구 TensorFlow Lite) 공식 문서 | https://developers.google.com/edge/litert/migration |

---

# 이번 주 제출물

1. 이 노트북 파일 (`.ipynb`) — [파일] → [다운로드] → [.ipynb 다운로드]
2. Teachable Machine **학습 완료 화면** 캡처 (클래스 이름과 사진 수가 보이도록)
3. `[실습 4-3]` 예측 결과 화면 캡처
4. `[과제 1]` 정확도 결과, `[과제 2]` 1~3번 답

**제출 기한 : 다음 수업 전날 오후 11:59까지 ｜ 제출 위치 : 팀즈에 2주차 과제 생성**

※ 파일 이름은 `날짜(YYYYMMDD)_이름_2주차.ipynb` 형식으로 바꿔서 제출하세요.
　(예: 20260825_홍길동_2주차.ipynb)

---

# 다음 주 예고 — 3주차

**Hugging Face 시작하기 — 남이 만든 AI 모델 가져다 쓰기**
오늘은 내가 직접 사진을 모아 학습시켰지만, 다음 주에는 **이미 학습이 끝난 모델**을
코드 세 줄로 불러와 문장의 감정을 분석해 봅니다.

> 준비물: Hugging Face 계정 (huggingface.co 에서 무료 가입, 3주차 전까지)
